# Quantum Teleportation

In [4]:
from pyquil.quil import Program
from pyquil.api import get_qc
from pyquil.gates import X, Z, H, CNOT

In [9]:
def make_bell_pair(q1, q2):
    """Makes a bell pair between qubits q1 and q2
    """
    return Program(H(q1), CNOT(q1, q2))


def teleport(start_index, end_index, ancilla_index):
    """Teleport a qubit from start to end using an ancilla qubit
    """
    program = make_bell_pair(end_index, ancilla_index)

    ro = program.declare("ro", memory_size=3)

    # do the teleportation
    program.inst(CNOT(start_index, ancilla_index))
    program.inst(H(start_index))

    # measure the results and store them in classical registers [0] and [1]
    program.measure(start_index, ro[0])
    program.measure(ancilla_index, ro[1])

    program.if_then(ro[1], X(2))
    program.if_then(ro[0], Z(2))

    program.measure(end_index, ro[2])

    #print(program)
    return program

In [10]:
qvm = get_qc("2q-qvm")
def run_and_read(prog, address, shots=1):
    """Resolve if_then labels, run on the QVM, return the bits stored at ro[address]."""
    prog = prog.copy()
    prog.resolve_label_placeholders()
    prog.wrap_in_numshots_loop(shots)
    return qvm.run(prog).get_register_map()["ro"][:, address]

# initialize qubit 0 in |1>
teleport_demo = Program(X(0))
teleport_demo += teleport(0, 2, 1)
print("Teleporting |1> state: {}".format(run_and_read(teleport_demo, 2)))

# initialize qubit 0 in |0>
teleport_demo = Program()
teleport_demo += teleport(0, 2, 1)
print("Teleporting |0> state: {}".format(run_and_read(teleport_demo, 2)))

# initialize qubit 0 in |+>
teleport_demo = Program(H(0))
teleport_demo += teleport(0, 2, 1)
print("Teleporting |+> state: {}".format(run_and_read(teleport_demo, 2, shots=10)))

Teleporting |1> state: [1]
Teleporting |0> state: [0]
Teleporting |+> state: [1 1 1 1 1 1 0 1 1 0]
